# Figure S6 — D0–D3 program annotation

This notebook loads the scientific inputs, selects features, rebuilds source tables, and renders the panels. The original selection and rendering functions are ported into `src/cc_perturbseq/figure_s06_annotations.py`; no previous notebook is opened, patched, executed, or copied.

Source: `program_annotation_panels_D0_D1_D2_D3_rotated_logos_D2_extra_D3_trimmed_markers.ipynb`, code cells 2, 4, 6, 8, 10, 12. Input paths now use the approved active min10 enrichment table, current cluster map, and standard compiled MoDISco H5. D2 extra genes, D3 trimmed markers, top-three motifs, shared scales, and rotated vector logos are retained.

In [ ]:
from pathlib import Path
import sys, os, json
from IPython.display import display
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "cc_perturbseq").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository checkout.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from cc_perturbseq.figure_notebook import initialize_figure, load_figure_manifest, get_path, panel_directory, write_receipt

from cc_perturbseq import figure_s06_annotations as s6
from cc_perturbseq.figure_s06_annotations import *
from cc_perturbseq.motif_sources import canonical_motif_input_manifest
ctx = initialize_figure(load_figure_manifest("Figure_S06"), require_inputs=False)
OUTDIR = panel_directory(ctx, "all")
SOURCE_DIR = OUTDIR / "_program_annotation_source"
# Only scientific data paths are inputs. There is no source-notebook path.
CONSENSUS_PATH = get_path(ctx["paths"], "ccperturbseq.cnmf_spectra")
PROGRAM_ANNOT_PATH = get_path(ctx["paths"], "ccperturbseq.program_annotations")
CNMF_WORKBOOK_PATH = REPO_ROOT / "data/figure_02/cNMF_50_0_2_032526_6co.xlsx"
motif_inputs = canonical_motif_input_manifest()
ENHANCER_ENRICHMENT_PATH = motif_inputs["enhancer_enrichment_results"]
CLUSTER_MAP_PATH = motif_inputs["cluster_motif_map"]
MODISCO_H5_PATH = motif_inputs["modisco_compiled_h5"]
input_paths = s6.configure_inputs(consensus=CONSENSUS_PATH, annotations=PROGRAM_ANNOT_PATH,
    workbook=CNMF_WORKBOOK_PATH, enrichment=ENHANCER_ENRICHMENT_PATH,
    cluster_map=CLUSTER_MAP_PATH, modisco=MODISCO_H5_PATH, output=OUTDIR)
display(input_paths)


## Load data, select regulators/markers/motifs and compute shared scales

In [ ]:

# ============================================================
# LOAD DATA, SELECT FEATURES, SAVE SOURCE TABLES
# ============================================================

program_meta = load_program_metadata()
pert = load_perturbation_associations()
enh = load_enhancer_enrichment()
consensus = read_consensus_matrix(CONSENSUS_PATH)

print("Program metadata:", program_meta.shape)
print("Perturb associations:", pert.shape)
print("Enhancer enrichment:", enh.shape)
print("Consensus matrix:", consensus.shape)

# choose program offset for consensus matrix
consensus_offset = choose_program_offset(consensus, program_meta)

program_meta.to_csv(SOURCE_DIR / "program_metadata_merged.csv", index=False)

day_packages = {}

for day in DAYS_TO_PLOT:
    meta_day = (
        program_meta.loc[program_meta["day"] == day]
        .sort_values("program_num")
        .reset_index(drop=True)
    )

    if meta_day.empty:
        print(f"[warning] no programs found for {day}")
        continue

    # regulators
    reg_by_prog, reg_union, reg_selected_df = select_top_regulators_for_programs(meta_day, pert)
    reg_effect_mat, reg_fdr_mat, reg_star_mat = build_regulator_matrices(meta_day, reg_union, pert)

    # markers
    marker_by_prog, marker_union, marker_selected_df = select_marker_genes_for_programs(
        program_meta_day=meta_day,
        consensus_df=consensus,
        consensus_offset=consensus_offset,
    )
    marker_loading_mat = build_marker_loading_matrix(meta_day, marker_union, consensus, consensus_offset)

    # motifs
    motif_by_prog, motif_union, motif_selected_df = select_top_motifs_for_programs(meta_day, enh)
    motif_value_mat, motif_qval_mat, motif_star_mat = build_motif_matrices(meta_day, motif_union, enh)

    # logos
    motif_to_logo, motif_logo_map_df = resolve_logo_paths_for_motifs(motif_union)

    day_packages[day] = {
        "meta": meta_day,
        "reg_union": reg_union,
        "reg_selected_df": reg_selected_df,
        "reg_effect_mat": reg_effect_mat,
        "reg_fdr_mat": reg_fdr_mat,
        "reg_star_mat": reg_star_mat,
        "marker_union": marker_union,
        "marker_selected_df": marker_selected_df,
        "marker_loading_mat": marker_loading_mat,
        "motif_union": motif_union,
        "motif_selected_df": motif_selected_df,
        "motif_value_mat": motif_value_mat,
        "motif_qval_mat": motif_qval_mat,
        "motif_star_mat": motif_star_mat,
        "motif_to_logo": motif_to_logo,
        "motif_logo_map_df": motif_logo_map_df,
    }

    # Save source tables
    prefix = SOURCE_DIR / f"{day}"

    meta_day.to_csv(f"{prefix}_programs.csv", index=False)

    reg_selected_df.to_csv(f"{prefix}_selected_regulators.csv", index=False)
    reg_effect_mat.to_csv(f"{prefix}_regulator_effect_matrix.csv", index_label="program_num")
    reg_fdr_mat.to_csv(f"{prefix}_regulator_fdr_matrix.csv", index_label="program_num")
    reg_star_mat.to_csv(f"{prefix}_regulator_star_matrix.csv", index_label="program_num")

    marker_selected_df.to_csv(f"{prefix}_selected_marker_genes.csv", index=False)
    marker_loading_mat.to_csv(f"{prefix}_marker_loading_matrix.csv", index_label="program_num")

    motif_selected_df.to_csv(f"{prefix}_selected_motifs.csv", index=False)
    motif_value_mat.to_csv(f"{prefix}_motif_enrichment_matrix.csv", index_label="program_num")
    motif_qval_mat.to_csv(f"{prefix}_motif_qval_matrix.csv", index_label="program_num")
    motif_star_mat.to_csv(f"{prefix}_motif_star_matrix.csv", index_label="program_num")
    motif_logo_map_df.to_csv(f"{prefix}_motif_logo_map.csv", index=False)

    print(f"\n=== {day} ===")
    print("Programs:", meta_day["program_num"].tolist())
    print("Selected regulators:", len(reg_union))
    print("Selected marker genes:", len(marker_union))
    print("Selected motifs:", len(motif_union))


# ============================================================
# SHARED D0-D3 COLOR SCALES
# ============================================================

shared_scales = compute_shared_color_scales(
    day_packages
)

shared_scale_values = {
    "regulator_vmin": -shared_scales["regulator_vmax"],
    "regulator_vmax": shared_scales["regulator_vmax"],
    "marker_vmin": 0.0,
    "marker_vmax": shared_scales["marker_vmax"],
    "motif_vmin": 0.0,
    "motif_vmax": shared_scales["motif_vmax"],
}

with open(
    SOURCE_DIR
    / "shared_color_scales_D0_D3.json",
    "w",
) as handle:
    json.dump(
        shared_scale_values,
        handle,
        indent=2,
    )


# ============================================================
# SAVE CONFIGURATION
# ============================================================

config_used = {
    "CONSENSUS_PATH": str(CONSENSUS_PATH),
    "PROGRAM_ANNOT_PATH": str(PROGRAM_ANNOT_PATH),
    "CNMF_WORKBOOK_PATH": str(CNMF_WORKBOOK_PATH),
    "ENHANCER_ENRICHMENT_PATH": str(
        ENHANCER_ENRICHMENT_PATH
    ),
    "CLUSTER_MAP_PATH": str(CLUSTER_MAP_PATH),
    "MODISCO_H5_PATH": str(MODISCO_H5_PATH),
    "PROGRAM_OFFSET": PROGRAM_OFFSET,
    "consensus_offset_chosen": consensus_offset,
    "N_TOP_REGULATORS": N_TOP_REGULATORS,
    "N_TOP_MOTIFS": N_TOP_MOTIFS,
    "D2_N_ADDITIONAL_TOP_LOADING_GENES": D2_N_ADDITIONAL_TOP_LOADING_GENES,
    "D3_N_CURATED_MARKERS_TO_KEEP": D3_N_CURATED_MARKERS_TO_KEEP,
    "D2_CURATED_GENES_FIRST": D2_CURATED_GENES_FIRST,
    "REGULATOR_FDR_THRESHOLD": (
        REGULATOR_FDR_THRESHOLD
    ),
    "MOTIF_QVAL_THRESHOLD": (
        MOTIF_QVAL_THRESHOLD
    ),
    "VECTOR_LOGOS": VECTOR_LOGOS,
    "ROTATE_MOTIF_LOGOS_DEGREES": (
        ROTATE_MOTIF_LOGOS_DEGREES
    ),
    "MAKE_PER_DAY_PANELS": (
        MAKE_PER_DAY_PANELS
    ),
    "MAKE_COMBINED_ALL_DAYS_FIGURE": (
        MAKE_COMBINED_ALL_DAYS_FIGURE
    ),
    "shared_color_scales": (
        shared_scale_values
    ),
}

with open(
    SOURCE_DIR
    / "figure_config_used.json",
    "w",
) as handle:
    json.dump(
        config_used,
        handle,
        indent=2,
    )

print(
    "\nSource tables written to:",
    SOURCE_DIR,
)

print(
    "Shared color scales written to:",
    SOURCE_DIR
    / "shared_color_scales_D0_D3.json",
)


## Render original-style per-day and combined panels

In [ ]:

# ============================================================
# RENDER PER-DAY PANELS AND COMBINED D0-D3 FIGURE
# ============================================================

if MAKE_PER_DAY_PANELS:
    print(
        "Rendering per-day panels with programs as rows, "
        "motifs as columns, and 90-degree rotated logos..."
    )

    for day in DAYS_TO_PLOT:
        if day not in day_packages:
            print(
                f"[skip] {day}: no package"
            )
            continue

        output_stem = make_per_day_panel(
            day=day,
            day_package=day_packages[day],
            shared_scales=shared_scales,
        )

        print(f"Saved {day}:")
        print(
            " ",
            output_stem.with_suffix(".svg"),
        )
        print(
            " ",
            output_stem.with_suffix(".pdf"),
        )
        print(
            " ",
            output_stem.with_suffix(".png"),
        )


if MAKE_COMBINED_ALL_DAYS_FIGURE:
    print(
        "\nRendering the combined D0-D3 figure "
        "with one shared colorbar per data type..."
    )

    combined_output_stem = (
        make_combined_all_days_figure(
            day_packages=day_packages,
            days=DAYS_TO_PLOT,
            shared_scales=shared_scales,
        )
    )

    print("Saved combined figure:")
    print(
        " ",
        combined_output_stem.with_suffix(
            ".svg"
        ),
    )
    print(
        " ",
        combined_output_stem.with_suffix(
            ".pdf"
        ),
    )
    print(
        " ",
        combined_output_stem.with_suffix(
            ".png"
        ),
    )


In [ ]:
generated = sorted(p for p in OUTDIR.rglob("*") if p.is_file())
write_receipt(ctx,"all",status="RESCRIPTED_SOURCE_RENDERED", inputs=input_paths,
    outputs={str(p.relative_to(OUTDIR)):str(p) for p in generated},
    parameters={"source_logic_changed":False,"runtime_source_notebook_required":False,
                "scientific_source_sha256":"b0ea563bee36408e54ff793e4915a7258232473a65813bba2bb5d7b5f655599a"})
print("Figure S6 outputs:", OUTDIR)
